# DS2002 · Data Science Systems Overview

**Lecture — 2026-08-26 · Fall 2026**  
**Class time:** 45 minutes

---

## The question this course is built around

A vendor outside Scott Stadium has to decide, on Thursday, how many rain ponchos to put on the truck for Saturday. Order too few and you sell out by the first quarter. Order too many and you eat the cost of storing them until next season.

You can answer that question two ways.

1. Open last season's spreadsheet, scroll around, and pick a number that feels right.
2. Build a short, written-down path from the sales records to the number — one you can re-run in October when you have four more games of data.

The second one is a **system**. Same inputs produce the same output, every step is visible, and next month's version costs you almost nothing because the work is already written down. That is the whole course: taking questions that people answer by hand and turning them into something repeatable.

### Demo 1 — the smallest pipeline that answers the question

Four home games. Clean numbers, on purpose, so we can see the shape of the work before the mess shows up.

In [1]:
import pandas as pd

games = pd.DataFrame({
    'game': ['Richmond', 'Coastal Carolina', 'NC State', 'Louisville'],
    'attendance': [41000, 38500, 47200, 44100],
    'ponchos_sold': [120, 95, 1450, 1610],
    'rain': [False, False, True, True],
})
games

,game,attendance,ponchos_sold,rain
0,Richmond,41000,120,False
1,Coastal Carolina,38500,95,False
2,NC State,47200,1450,True
3,Louisville,44100,1610,True


Raw poncho counts are not comparable across games — a sellout crowd should move more of everything. Convert to a **rate** first, then compare wet games to dry ones.

In [2]:
games['ponchos_per_1k'] = games['ponchos_sold'] / (games['attendance'] / 1000)
rate = games.groupby('rain')['ponchos_per_1k'].mean().round(1)
rate

,ponchos_per_1k
rain,
False,2.7
True,33.6


In [3]:
# Turn the rate into the actual decision: how many to load on the truck?
expected_attendance = 45_000

for wet, r in rate.items():
    label = 'rain in the forecast' if wet else 'dry forecast'
    print(f'{label}: stock about {round(r * expected_attendance / 1000):,} ponchos')

dry forecast: stock about 122 ponchos
rain in the forecast: stock about 1,512 ponchos


That is a complete pipeline: **ingest -> derive -> group -> decide.** Six lines, and it answers a real question. Hold onto how clean it felt, because that is not what the data looks like.

### Demo 2 — the same question, with the data you actually get

This is the export the vendors send. Same four games, same ponchos. Look at what happened to it on the way here.

In [4]:
from io import StringIO

raw = '''game,item,units,unit_price,attendance
Richmond,Rain Poncho,120,$6.00,41000
Coastal Carolina,rain poncho,95,6.00,38500
NC State,Rain Poncho ,1450,$6.00,47200
NC State,Rain Poncho ,1450,$6.00,47200
Louisville,PONCHO,1610,6,44100
Louisville,Rain Poncho,-40,6,44100'''

sales = pd.read_csv(StringIO(raw))
sales

,game,item,units,unit_price,attendance
0,Richmond,Rain Poncho,120,$6.00,41000
1,Coastal Carolina,rain poncho,95,6.00,38500
2,NC State,Rain Poncho,1450,$6.00,47200
3,NC State,Rain Poncho,1450,$6.00,47200
4,Louisville,PONCHO,1610,6,44100
5,Louisville,Rain Poncho,-40,6,44100


Five problems in six rows: the same product is written four different ways, the NC State row was exported twice, one row is a refund entered as negative units, and `unit_price` is text because somebody's system printed a dollar sign.

Ask the naive question anyway and watch what you get.

In [5]:
# 'How many ponchos did we sell?' — the version that looks reasonable
print(sales.groupby('item')['units'].sum())
print()
print('naive total:', sales['units'].sum())

item
PONCHO          1610
Rain Poncho       80
Rain Poncho     2900
rain poncho       95
Name: units, dtype: int64

naive total: 4685


Four product lines that are all the same poncho, and a total that quietly includes the duplicated NC State row. Now clean it — three decisions, each one written down.

In [6]:
clean = sales.drop_duplicates().copy()          # the double-exported row
clean['item'] = clean['item'].str.strip().str.lower()   # one spelling
clean = clean[clean['units'] > 0]               # refunds are not sales

print(clean.groupby('item')['units'].sum())
print()
print('cleaned total:', clean['units'].sum())

item
poncho         1610
rain poncho    1665
Name: units, dtype: int64

cleaned total: 3275


In [7]:
naive = sales['units'].sum()
real = clean['units'].sum()
print(f'naive:   {naive:,}')
print(f'cleaned: {real:,}')
print(f'overstated by {naive - real:,} ponchos '
      f'= ${(naive - real) * 6:,.0f} of stock you did not need')

naive:   4,685
cleaned: 3,275
overstated by 1,410 ponchos = $8,460 of stock you did not need


The cleaning did not make the report prettier. It changed the answer by about 40 percent and roughly eight thousand dollars of inventory. That gap is the reason this course spends real time on cleaning, joining, and validation instead of jumping to charts.

### Making it repeatable

Right now those three cleaning decisions live in a cell someone can edit by accident. Put them in a function and the pipeline becomes something you can trust and re-run.

In [8]:
def poncho_units(df):
    """Units sold per game, after dropping duplicate exports, normalizing the
    product name, and removing refund rows."""
    out = df.drop_duplicates().copy()
    out['item'] = out['item'].str.strip().str.lower()
    out = out[out['units'] > 0]
    return out.groupby('game')['units'].sum()

first = poncho_units(sales)
second = poncho_units(sales)

print(first)
print()
print('same answer both times:', first.equals(second))

game
Coastal Carolina      95
Louisville          1610
NC State            1450
Richmond             120
Name: units, dtype: int64

same answer both times: True


Two properties came free with that function, and both matter more than they look:

- **Repeatable.** Run it twice, get the same answer. Run it in November on twelve games instead of four and it still works.
- **Reviewable.** Someone can read those three lines and argue with them. "Why did you drop negative units instead of netting them against sales?" is a real question, and the function is where you have that argument.

Everything we build this semester ends in a function or a query someone else could read.

### Where the semester goes

Each block adds one capability to that same loop:

| Weeks | What you add |
|---|---|
| 2 | Git and GitHub, so work has history and a place to live |
| 3–4 | SQL and SQLite, then the same operations in Pandas |
| 5 | Cleaning as a discipline: types, duplicates, missing values, text |
| 6–7 | JSON and live APIs — data that arrives nested and sometimes fails |
| 8 | ETL end to end, framed by the Walmart hurricane case |
| 9–11 | Midterm: messy Walmart sales joined to real weather |
| 12 | Charts that carry an argument |
| 13–16 | Capstone: Game Day Pulse, start to recommendation |

By the midterm you will be doing today's demo on a hundred thousand rows with a live weather API in the middle of it.

### How the week runs

- **Monday** — lecture like today: concept plus a demo you run alongside me.
- **Wednesday** — studio. Hands-on building, ending with a short checkpoint you submit in Canvas for participation.
- **Friday** — the lab. This is the graded work for the week, and it is meant to take real time. There is no separate homework and no final exam.

Where things live:

- **Kaggle or Colab** — everything runs in the browser. Nothing to install.
- **GitHub** — course notebooks and materials, and where your own work gets versioned.
- **Canvas** — assignments, due dates, and grades. Announcements there are the official record, so read them.
- **Discord** (DS2002F26) — where questions get answered during the week. Post in the questions forum rather than DMing, so the answer helps whoever hits the same wall next. Anything personal goes to email.

### Practice 1 — revenue, not just units

`unit_price` came in as text because of the dollar signs. Turn it into a number, add a `revenue` column, and print revenue per game.

Expected: four games, Louisville highest.

In [16]:
work = clean.copy()
# 1. strip the '$' and convert to float
work['unit_price'] = work['unit_price'].astype(str).str.replace('$', '', regex=False).astype(float)
# 2. revenue = units * unit_price
work['revenue'] = work['units'] * work['unit_price']
# 3. print revenue per game
print(work[['game','revenue']])


               game  revenue
0          Richmond    720.0
1  Coastal Carolina    570.0
2          NC State   8700.0
4        Louisville   9660.0


### Practice 2 — which game ran hottest?

Using `work`, compute ponchos sold per 1,000 attendance for each game and print the game with the highest rate. Does it match what you would have guessed from raw units?

In [22]:
# rate per 1,000 attendance, then the top game
work['ponchos_per_1k'] = work['units'] / (work['attendance'] / 1000)

# print the game name with the highest rate
print(work.loc[work['ponchos_per_1k'].idxmax(), 'game'])




Louisville


### Before Friday

Friday's lab sets up the two things you need all semester: a working browser notebook and a GitHub repository of your own. Get a free GitHub account created before then if you do not have one — that is the only prerequisite.